# 📘 SFT vs. DPO vs. RL: The Practical Decision Framework & Case Studies

## 🎯 Purpose of this Notebook
Many engineers understand the *mechanics* of fine-tuning, but struggle with the architectural decision:
> **"When should I use standard Fine-Tuning (SFT)? When do I need DPO? And when should I reach for Reinforcement Learning (RL)?"**

This notebook provides:
1. **The Intuitive Mental Model** (zero jargon, practical analogies).
2. **4 Tested/Testable Industry Case Studies** with concrete code setups.
3. **Interactive Demonstrations** simulating data formats, loss behaviors, and evaluation rubrics for all 3 paradigms.
4. **A Production Decision Tree** to select the right approach for any real-world AI project.

---
## 🧠 1. The Core Mental Model

| Paradigm | The Core Command | Real-World Analogy | How Loss Works | What It Can't Do |
| :--- | :--- | :--- | :--- | :--- |
| **SFT (Supervised Fine-Tuning)** | *"Copy this exact answer."* | Reading a textbook and copying the answer key. | Maximizes probability of ground-truth tokens: $\max \log P(y \mid x)$ | **Cannot penalize bad behavior** (can't teach "don't do this"). |
| **DPO (Direct Preference Optimization)** | *"Pick Option A, never do Option B."* | A teacher crossing out the bad answer with a red pen. | Increases odds of $y_w$ while decreasing odds of $y_l$: $\sigma(r_w - r_l)$ | Requires humans/models to write paired bad examples beforehand. |
| **RL (Reinforcement Learning / GRPO)** | *"Figure it out yourself. If tests pass, you get a treat."* | Solving a maze or writing code until unit tests pass. | Samples multiple outputs, scores via automated verifier, reinforces winner. | Only works when success can be measured by a computer/verifier. |

---
## 📦 Setup & Environment Inspection

In [ ]:
import json
import re
from typing import Dict, Any, List, Tuple

print("✅ Framework libraries ready. No heavy GPU requirements needed for this walkthrough.")

---
## 🏢 Case Study 1: The E-Commerce Structured JSON Extractor
### **Problem**: Extracting unstructured order chat into strict JSON schema.
### **Decision**: Start with **SFT**, follow with **DPO** to prune chattiness.

```
Input: "John ordered 3 laptops for $2400, deliver Friday."
Goal:  {"customer": "John", "quantity": 3, "product": "laptops", "amount": 2400.0, "delivery_day": "Friday"}
```

### Why SFT is necessary first:
The base model doesn't know your company's schema keys (`customer`, `quantity`, `product`, `amount`, `delivery_day`). SFT teaches the knowledge and schema structure.

### Why DPO is needed second:
Even after SFT, on ~8-15% of inputs, models prepend conversational chattiness (`"Sure! Here is the JSON:..."`) or markdown fences (````json ... ````). SFT cannot unlearn this because SFT only maximizes positive tokens. DPO actively suppresses the chatty tokens.

In [ ]:
# 1. SFT Data Format (Single Positive Target)
sft_sample = {
    "prompt": "Extract order info: John ordered 3 laptops for $2400, deliver Friday.",
    "completion": '{"customer": "John", "quantity": 3, "product": "laptops", "amount": 2400.0, "delivery_day": "Friday"}'
}

# 2. DPO Data Format (Pairwise: Chosen vs. Rejected)
dpo_sample = {
    "prompt": "Extract order info: John ordered 3 laptops for $2400, deliver Friday.",
    "chosen": '{"customer": "John", "quantity": 3, "product": "laptops", "amount": 2400.0, "delivery_day": "Friday"}',
    "rejected": 'Sure! Here is the extracted JSON order details:\n```json\n{"customer": "John", "quantity": 3, "product": "laptops", "amount": 2400.0, "delivery_day": "Friday"}\n```\nHope that helps!'
}

print("=== Case Study 1 Comparison ===")
print("[SFT teaches]: What keys exist and how values map.")
print("[DPO teaches]: Strip the preamble 'Sure! Here is...' and backticks entirely.")

---
## 💻 Case Study 2: Code Generation & Bug Fixing
### **Problem**: Writing Python functions that must execute without runtime crashes.
### **Decision**: Use **Reinforcement Learning (GRPO / PPO)** with automated unit tests as the reward.

### Why SFT struggles alone:
You can train an SFT model on 100,000 GitHub code snippets. The model learns syntax, but it writes code that *looks* right to humans while failing on edge cases (e.g. empty lists, negative inputs, off-by-one errors).

### Why RL is the silver bullet here:
We don't need a human to write good and bad examples. **The Python interpreter itself is the judge!** We run automated unit tests against candidate generations and reward the code that passes all tests.

In [ ]:
# Demonstrating an Automated Reward Function for RL (GRPO-style verifier)
def code_reward_verifier(generated_code_str: str) -> float:
    """Automated unit test verifier acting as an RL reward function."""
    score = 0.0
    
    # Test 1: Syntax check
    try:
        compile(generated_code_str, "<string>", "exec")
        score += 0.2  # Syntax compiles successfully
    except SyntaxError:
        return 0.0
    
    # Test 2: Execution & logic check in safe local scope
    scope = {}
    try:
        exec(generated_code_str, scope)
        fn = scope.get("reverse_words")
        if fn:
            # Unit test 1: Standard case
            if fn("hello world") == "world hello":
                score += 0.4
            # Unit test 2: Edge case (multiple spaces)
            if fn("  deep learning  ") == "learning deep":
                score += 0.4
    except Exception:
        pass
        
    return score

# Candidate generations from the model during RL exploration:
candidate_A = "def reverse_words(s): return ' '.join(s.split()[::-1])"  # Perfect
candidate_B = "def reverse_words(s): return ' '.join(s.split(' ')[::-1])"  # Fails multiple spaces
candidate_C = "def reverse_words(s): return reversed(s)"  # Syntax error / wrong type

print("=== Case Study 2: RL Reward Verifier in Action ===")
print(f"Candidate A Reward: {code_reward_verifier(candidate_A)} / 1.0 (Favored by RL)")
print(f"Candidate B Reward: {code_reward_verifier(candidate_B)} / 1.0 (Partial Reward)")
print(f"Candidate C Reward: {code_reward_verifier(candidate_C)} / 1.0 (Penalized)")

---
## ✈️ Case Study 3: Brand Voice & Customer Service Tone
### **Problem**: Handling furious airline passengers whose flights were canceled.
### **Decision**: Use **DPO** (Human Preference Alignment).

### Why SFT fails here:
SFT mimics whatever is in the training set. If a customer says *"I will sue you, you ruined my wedding!"*, SFT often generates robotic corporate speak:
> *"Thank you for contacting airline support! We appreciate your patience. Flights can sometimes be delayed due to operational requirements."*

### Why RL cannot be used here:
**You cannot write a Python unit test for "empathy" or "brand warmth".** There is no compiler or formula to give a numeric score.

### Why DPO is the perfect fit:
Customer service leads review 300 real customer emails and label preferred vs. rejected replies. DPO directly optimizes human stylistic preference.

In [ ]:
cs_dpo_sample = {
    "prompt": "User: My flight was canceled 1 hour before departure. I am missing my sister's wedding! Fix this now or I sue!",
    "chosen": (
        "I am truly sorry. Missing your sister's wedding is awful, and I completely understand your anger. "
        "I have immediately pulled up your reservation (PNR: #AB123). Flight 408 departs in 2 hours with open seats, "
        "or I can book you on Partner Airline Flight 12 at 4:30 PM. Which one should I confirm for you right now?"
    ),
    "rejected": (
        "Thank you for reaching out to customer care! We deeply apologize for any inconvenience caused. "
        "Please be advised that per our carriage agreement section 4.2, flight schedules are not guaranteed. "
        "Please file a complaint on our web portal www.airline.com/complaints to request a travel voucher in 7-10 business days."
    )
}

print("=== Case Study 3: Tone Alignment (DPO) ===")
print("Chosen (Human verified empathy + actionable solution):")
print(cs_dpo_sample["chosen"][:140] + "...")
print("\nRejected (Robotic corporate dismissal):")
print(cs_dpo_sample["rejected"][:140] + "...")

---
## 🏥 Case Study 4: Medical / Financial Extraction (Anti-Hallucination)
### **Problem**: Summarizing doctor notes without inventing medications or symptoms.
### **Decision**: **Two-Stage Pipeline** (SFT for medical terminology $\to$ DPO for negative hallucination penalty).

### The Workflow:
1. **Stage 1 (SFT)**: Train model on clinical shorthand (`"Pt c/o SOB x 3d, PMHx HTN, DM2"` $\rightarrow$ Structured patient history). The model must learn the medical vocabulary first.
2. **Stage 2 (DPO)**: When testing on patient records, models sometimes hallucinate common medications (e.g., assuming a diabetic patient is on Metformin even if the chart didn't say so).
   - **Chosen ($y_w$)**: Summary containing *only* stated facts.
   - **Rejected ($y_l$)**: Summary containing inferred/hallucinated medications.
   - DPO heavily penalizes hallucinated entities.

In [ ]:
def hallucination_detector(source_text: str, summary_text: str, allowed_entities: List[str]) -> bool:
    """Simple check verifying summary does not introduce unmentioned entities."""
    source_lower = source_text.lower()
    summary_lower = summary_text.lower()
    
    for entity in allowed_entities:
        if entity.lower() in summary_lower and entity.lower() not in source_lower:
            return True  # Hallucinated!
    return False

source = "Patient has hypertension. Blood pressure 145/90. Prescribed Lisinopril 10mg daily."
good_summary = "Hypertension diagnosed, BP 145/90, Lisinopril 10mg started."
bad_summary = "Hypertension and type 2 diabetes diagnosed, BP 145/90, Lisinopril and Metformin started."

print("=== Case Study 4: Hallucination Verification ===")
print("Good summary hallucinated?:", hallucination_detector(source, good_summary, ["diabetes", "metformin"]))
print("Bad summary hallucinated? :", hallucination_detector(source, bad_summary, ["diabetes", "metformin"]))
print("-> DPO is trained using Good as Chosen, Bad as Rejected to suppress hallucination likelihood.")

---
## 🧭 5. The Production Decision Tree (Save This)

```
                                START: What is your primary objective?
                                                  │
             ┌────────────────────────────────────┴────────────────────────────────────┐
             ▼                                                                         ▼
   [NEW KNOWLEDGE / FORMAT]                                                  [BEHAVIOR & QUALITY]
Does the model already know your                                          Can a computer script verify if
vocabulary, schema, or task?                                              an output is objectively correct?
             │                                                                         │
     ┌───────┴───────┐                                                         ┌───────┴───────┐
     ▼               ▼                                                         ▼               ▼
   [NO]            [YES]                                                     [YES]            [NO]
Use SFT         Move to Behavior                                           Use RL           Use DPO
(Fine-Tuning)   Evaluation.                                              (GRPO/PPO)       (Preferences)
                                                                         • Math           • Tone & Style
                                                                         • Code/SQL       • Conciseness
                                                                         • Compilers      • Anti-chatter
                                                                         • Schema tests   • Empathy
```

### Summary Checklist for Technical Interviews:
1. **"Why not just use SFT for everything?"**
   * *Answer*: SFT is pure imitation learning (maximum likelihood). It cannot teach negative constraints ("don't do X"), and it suffers from exposure bias on out-of-distribution inputs.
2. **"Why use DPO over classic RLHF (PPO)?"**
   * *Answer*: DPO mathematically re-parameterizes the reward model into the language model's own probabilities. It eliminates the separate Reward Model and Critic network, saving massive GPU VRAM and eliminating training instability.
3. **"When is RL (like DeepSeek's GRPO) better than DPO?"**
   * *Answer*: When there is an objective, automated verifier (like unit tests or math checkers). RL lets the model self-explore and discover novel reasoning strategies without requiring humans to hand-write pairs of good and bad answers.